# QTS Final Project: Technical Study (Draft)

## Crypto Perpetual Futures Market Neutral Strategy

**Team Members**: Samos Zhu (12498135), Wenxin Chang (12497798), Henry Xu (12284734), Cody Torgovnik (12496679)

## 1. Project Idea – Exposition

### 1.1 Core Concept

Crypto perpetual futures settle funding every 8 hours (00:00, 08:00, 16:00 UTC): longs and shorts exchange a payment based on the **funding rate**. When funding is positive, longs pay shorts; when negative, shorts pay longs. This creates a carry-like cash flow that we aim to harvest while staying **market-neutral**.

### 1.2 Alpha Logic

1. **Funding Carry**: Long assets with low/negative funding (receive funding); short assets with high funding (receive funding).
2. **Crowding Filter**: High funding + rising OI/volume → crowded longs → fade by shorting. Low funding + not crowded → potential longs. Since OI is not available on data.binance.vision, we use **volume change** as a crowding proxy.
3. **Risk Overlay**: Target-vol scaling and drawdown gates limit leverage; a 5-minute tail brake reduces exposure on sharp intraday moves.

### 1.3 Strategy Structure

- **8h Alpha**: Rebalance Long Bottom-K / Short Top-K by funding + crowding signal.
- **1h Risk**: Target-vol scaling, drawdown gate (>8% halve, >12% flat).
- **5m Tail**: Event brake on sharp moves (±2% / ±3% triggers).
- **Universe**: BTC, ETH, SOL, BNB, XRP, ADA, DOGE, AVAX (8 USDT perpetuals).
- **Period**: 2022–2024 (bear, chop, bull regimes).

## 2. Data: Obtaining, Cleaning, Formatting

Data is obtained from **data.binance.vision** (static zip downloads, no REST API). Run `doit` first to fetch and align data; the notebook then loads from disk.

**Note**: If data does not exist, the cell below will attempt to fetch it (takes 30–60 min for full 8 symbols, 3 years).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from src.config import DATA_DIR, START_DATE, END_DATE
from src.data_fetch import build_local_dataset
import pandas as pd

# Obtain data (skip if already fetched via doit)
if not (DATA_DIR / 'klines_1h.parquet').exists():
    build_local_dataset({'start': START_DATE, 'end': END_DATE}, DATA_DIR)
else:
    print('Data exists in', DATA_DIR, '– using pre-fetched data')

In [ ]:
# Load raw data
klines_1h = pd.read_parquet(DATA_DIR / 'klines_1h.parquet')
klines_5m = pd.read_parquet(DATA_DIR / 'klines_5m.parquet')
funding = pd.read_parquet(DATA_DIR / 'funding_8h.parquet')

print('Klines 1h:', klines_1h.shape)
print('Klines 5m:', klines_5m.shape)
print('Funding:', funding.shape)
klines_1h.head()

In [ ]:
# Clean and format: align to 1h panel (UTC, funding at settlement hours)
from src.align import align_to_panel

panel = align_to_panel(DATA_DIR)
panel['ts'] = pd.to_datetime(panel['ts'])
panel = panel.sort_values(['ts', 'symbol'])

print('Panel shape:', panel.shape)
print('Columns:', panel.columns.tolist())
panel.head(10)

## 3. Data Graphs

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Graph 1: Normalized close prices (sample symbols)
fig, ax = plt.subplots(figsize=(10, 4))
for sym in ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']:
    sub = panel[panel['symbol'] == sym].set_index('ts')['close']
    ax.plot(sub.index, sub / sub.iloc[0], label=sym.replace('USDT', ''))
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Close')
ax.set_title('Graph 1: Normalized Close Prices (Sample Symbols)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 2: Funding rate (BTCUSDT, basis points)
fig, ax = plt.subplots(figsize=(10, 4))
fb = funding[funding['symbol'] == 'BTCUSDT'].copy()
fb['ts'] = pd.to_datetime(fb['ts'])
ax.bar(fb['ts'], fb['funding_rate'] * 1e4, width=0.3, alpha=0.8)
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Funding Rate (bps)')
ax.set_title('Graph 2: BTCUSDT Funding Rate')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 3: Average hourly volume by symbol
fig, ax = plt.subplots(figsize=(8, 4))
vol = panel.groupby('symbol')['volume'].mean().sort_values(ascending=True)
ax.barh(vol.index.str.replace('USDT', ''), vol.values)
ax.set_xlabel('Average Hourly Volume')
ax.set_ylabel('Symbol')
ax.set_title('Graph 3: Average Hourly Volume by Symbol')
plt.tight_layout()
plt.show()

## 4. Summary

- Data from data.binance.vision (klines + funding).
- Panel aligned to 1h grid: ret_1h, funding_rate, volume_change (crowding proxy), is_alpha_time.
- Crowding: volume_change = 成交量变化 (OI 不好爬取，用 volume pct_change 替代).
- Performance analysis and robustness to follow in final submission.